# Introduction

In this notebook we will try to apply the newest Google's Perch model to identify the birds, insects, mammals and amphibians. Unfortunately, this model is too slow, and it was impossible (at least for me, but you can do better) to make the predictions for all the provided files. So, only a part of them is processed. This makes it possible to estimate the overall score of the model.

# Import common libraries

Note: we need tensorflow-2.20 at least, and tensorflow-2.19 that is default in kaggle environment for now, produces error!

In [1]:
!pip install /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl
!pip install /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Processing /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.
Processing /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behav

In [2]:
import time
START = time.time()

from concurrent.futures import ThreadPoolExecutor
import glob
import librosa
import numpy as np
import os
import pandas as pd
import re
import sys
import tensorflow as tf

tf.experimental.numpy.experimental_enable_numpy_behavior()

TERMINATE_TIME = START + 5300

# Import the model
birdclassifier = tf.saved_model.load('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/')

2026-03-11 22:08:26.966001: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Species mapping

Perch is intended to predict *a lot of* species, while we are working with just few of them here. It is important to get the predictions in the right order.

In [3]:
bc_labels = pd.read_csv('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1/assets/labels.csv')
print(bc_labels)

                inat2024_fsd50k
0            Abavorana luctuosa
1             Abeillia abeillei
2        Abroscopus albogularis
3        Abroscopus schisticeps
4      Abroscopus superciliaris
...                         ...
14790    Zosterornis whiteheadi
14791           Zulpha perlaria
14792       Zvenella geniculata
14793       Zvenella transversa
14794          Zvenella yunnana

[14795 rows x 1 columns]


We can see, that Perch's labels are in scientific format. So we need a common name for each of the species we work with. Luckly, we have taxonomy data.

In [4]:
bc_labels = bc_labels.reset_index().rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1).set_index('scientific_name')
taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')

mapping = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping.bc_index = mapping.bc_index.fillna(len(bc_labels)).astype(int)
mapping = mapping[['primary_label', 'bc_index']].set_index('primary_label')

primary_labels = pd.read_csv('/kaggle/input/competitions/birdclef-2026/sample_submission.csv').columns[1:].to_list()
birdclassifier_indices = [int(mapping.loc[pl][0]) for pl in primary_labels]
birdclassifier_indices[:5]

/tmp/ipykernel_17/251601943.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  birdclassifier_indices = [int(mapping.loc[pl][0]) for pl in primary_labels]


[5743, 14795, 7018, 14795, 14795]

In [5]:
total_predicted_species = len(set(birdclassifier_indices) - {len(bc_labels)})
print(f'Note: we can predict {total_predicted_species} out of 234 species only!')

Note: we can predict 203 out of 234 species only!


# Get the data files

In [6]:
def get_oggs(max_oggs=8):
    if len(glob.glob('/kaggle/input/competitions/birdclef-2026/test_soundscapes/*.ogg')) > 0:
        oggs = glob.glob('/kaggle/input/competitions/birdclef-2026/test_soundscapes/*.ogg')
    else:
        oggs = sorted(glob.glob(f'/kaggle/input/competitions/birdclef-2026/train_soundscapes/*.ogg'))[:max_oggs]
    return [(n, ogg, re.search(r'/([^/]+)\.ogg$', ogg).group(1), n < int(len(oggs))) for n, ogg in enumerate(oggs)]

oggs = get_oggs()

# Process the files in threads

In [7]:
def perch_result(ogg):
    _, fname, ss_id, do_prediction = ogg
    sr = 32_000

    print(f'{ss_id}')
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]

    if not do_prediction or time.time() > TERMINATE_TIME:
        return row_ids, -1000 * np.ones((12, len(primary_labels)))
        
    data, _ = librosa.load(fname, sr=sr)
    model_outputs = birdclassifier.signatures['serving_default'](inputs=data.reshape((-1, 5 * sr)))['label']

    model_outputs = tf.pad(model_outputs, tf.constant([[0, 0], [0, 1]]))
    result = model_outputs[:, birdclassifier_indices]

    return row_ids, result

In [8]:
row_ids = []
result = []

with ThreadPoolExecutor(max_workers=4) as executor:
    for ogg_row_ids, ogg_result in executor.map(perch_result, oggs):
        row_ids += ogg_row_ids
        result.append(ogg_result)

submission = pd.DataFrame(np.concatenate(result), columns=primary_labels)
submission['row_id'] = row_ids
submission = submission[['row_id'] + primary_labels]

BC2026_Train_0001_S08_20250606_030007
BC2026_Train_0002_S08_20250607_030007
BC2026_Train_0003_S08_20250607_070007
BC2026_Train_0004_S08_20250607_070007


I0000 00:00:1773266935.770679      66 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


BC2026_Train_0005_S08_20250607_070007
BC2026_Train_0006_S09_20250828_000000
BC2026_Train_0007_S09_20250829_000000BC2026_Train_0008_S09_20250831_000000



# Make a submission

In [9]:
submission.to_csv('submission.csv', index=False)

# Display submission DataFrame
display(submission.head(20))

display(submission.tail(20))

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,-0.254748,0.0,-1.381532,0.0,0.0,0.952681,-1.736017,-0.520771,-2.373590,...,-0.686290,3.506053,0.723470,1.008493,0.025720,3.198991,-0.434282,-0.228001,1.416989,2.898700
1,BC2026_Train_0001_S08_20250606_030007_10,0.065003,0.0,-1.654960,0.0,0.0,1.052379,-1.978870,-0.375204,-2.573109,...,-0.225047,3.343784,1.635845,1.341101,-1.111406,1.928457,-0.322878,-0.362796,1.946862,2.741418
2,BC2026_Train_0001_S08_20250606_030007_15,-0.028887,0.0,-1.535252,0.0,0.0,0.858210,-1.866930,-0.707474,-2.492187,...,-0.091466,3.244307,1.193571,1.358101,-0.704286,2.754735,-0.288283,-0.150839,1.707637,1.867248
3,BC2026_Train_0001_S08_20250606_030007_20,0.285777,0.0,-1.270706,0.0,0.0,1.087190,-1.740027,-0.382811,-2.429112,...,0.224674,3.368449,0.590748,0.756146,-0.447998,2.656746,-0.255418,0.080333,1.805441,2.213743
4,BC2026_Train_0001_S08_20250606_030007_25,0.519409,0.0,-1.572041,0.0,0.0,0.837248,-1.894629,-0.442360,-2.342682,...,-0.488339,3.493846,0.869774,0.810443,-1.058075,2.660435,-0.323529,-0.136607,2.005119,3.062678
5,BC2026_Train_0001_S08_20250606_030007_30,-0.047996,0.0,-1.368970,0.0,0.0,0.841419,-1.713940,-0.504282,-2.156103,...,-0.605979,4.218234,1.387794,1.304804,-0.056016,3.687824,-0.185664,-0.606628,2.550937,3.116639
6,BC2026_Train_0001_S08_20250606_030007_35,-0.126365,0.0,-1.515226,0.0,0.0,0.611219,-0.180519,0.505977,-2.007793,...,-0.418463,2.952921,1.074697,1.701145,-0.846160,3.003455,0.743171,-0.467751,2.533823,3.169537
7,BC2026_Train_0001_S08_20250606_030007_40,0.056745,0.0,-1.610670,0.0,0.0,1.333731,-2.055432,-0.832298,-2.704738,...,-0.907773,2.814581,0.734805,1.000151,-1.262714,1.535165,-0.226494,-0.240743,1.895455,1.814024
8,BC2026_Train_0001_S08_20250606_030007_45,0.352309,0.0,-1.125197,0.0,0.0,0.610802,-1.782815,-0.140241,-2.243804,...,-0.284377,2.929173,1.172981,1.675218,-0.739893,2.420005,-0.381794,-0.425887,2.198058,2.874241
9,BC2026_Train_0001_S08_20250606_030007_50,0.016101,0.0,-1.768335,0.0,0.0,0.973278,-1.446398,-0.554233,-1.924692,...,-0.396923,2.990757,0.648377,1.023082,-0.822751,2.277219,-0.053785,-0.006588,2.011724,3.836250


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
76,BC2026_Train_0007_S09_20250829_000000_25,-2.060732,0.0,-1.552045,0.0,0.0,0.669277,-1.059449,-0.591419,-2.682223,...,0.225070,3.102250,0.480101,0.899442,-0.453890,1.706635,-1.186401,0.188331,4.201413,2.617743
77,BC2026_Train_0007_S09_20250829_000000_30,-1.778312,0.0,-1.435296,0.0,0.0,3.038978,-0.623080,-1.078736,-2.690852,...,0.095147,2.262340,0.306325,1.845431,-1.185442,1.343289,-0.875851,-0.294173,1.729430,1.877227
78,BC2026_Train_0007_S09_20250829_000000_35,-1.748044,0.0,-1.667085,0.0,0.0,0.432222,-1.265039,-1.069904,-2.567740,...,0.329276,3.307281,0.630981,2.787348,-0.292853,1.421427,-0.129044,1.026122,2.102557,2.765224
79,BC2026_Train_0007_S09_20250829_000000_40,-1.631879,0.0,-1.652521,0.0,0.0,1.067378,-1.297857,-0.319872,-2.545804,...,-0.885153,5.394797,1.096167,2.431193,-0.318389,2.462068,2.635440,-0.941176,4.575734,2.846022
80,BC2026_Train_0007_S09_20250829_000000_45,-2.067602,0.0,-1.649136,0.0,0.0,1.349023,-1.031312,-1.417668,-2.708756,...,0.301810,3.982993,0.638625,1.563825,0.220335,1.278688,0.456832,0.057593,2.790352,2.505362
81,BC2026_Train_0007_S09_20250829_000000_50,-2.030624,0.0,-1.593999,0.0,0.0,1.910439,-1.500306,-0.660373,-2.735613,...,-0.929819,5.827284,0.889732,1.572368,0.102556,3.938596,5.537585,-0.429450,4.168178,2.691462
82,BC2026_Train_0007_S09_20250829_000000_55,-1.406322,0.0,-1.524417,0.0,0.0,1.494923,-0.952812,-0.233159,-2.624441,...,1.867952,3.918900,-0.118289,1.603308,-0.508344,1.247596,-0.041228,0.700943,5.454757,2.242199
83,BC2026_Train_0007_S09_20250829_000000_60,-1.585755,0.0,-1.484080,0.0,0.0,0.866986,-1.257575,-0.495901,-2.692999,...,0.474789,4.587723,0.614893,3.222436,2.163164,3.279823,2.522323,1.079236,4.113927,2.880625
84,BC2026_Train_0008_S09_20250831_000000_5,-2.661673,0.0,-1.405025,0.0,0.0,0.542999,-1.773856,-0.997765,-2.703430,...,-0.042356,3.124086,0.220786,1.664632,-0.624576,0.741972,-1.404191,-0.796347,2.499798,3.092195
85,BC2026_Train_0008_S09_20250831_000000_10,-2.714342,0.0,-0.884152,0.0,0.0,1.198545,-1.512897,-0.043567,-2.473316,...,0.041161,3.153775,-0.258805,0.937860,-0.225919,1.498759,-0.772352,0.472700,2.473800,2.311219
